<a href="https://colab.research.google.com/github/cherrynii/cherrynii.github.io/blob/main/Chromatin_State_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from google.colab import drive
from keras import regularizers
from sklearn.model_selection import train_test_split
from keras.losses import CategoricalCrossentropy

In [ ]:
# Importing the data from Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
sequences = pd.read_csv("/content/drive/MyDrive/cbshackathon/trainsequences.csv", header=None)
labels = pd.read_csv("/content/drive/MyDrive/cbshackathon/trainlabels.csv", header=None)
test = pd.read_csv("/content/drive/MyDrive/cbshackathon/testsequences.csv", header=None)
print(sequences.shape)
print(labels.shape)
print(test.shape)

(286164, 1)
(286164, 1)
(100008, 1)


In [ ]:
# Adding reverse complement sequences

def reverse_complement(seq):
    complement = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C'
    }
    return ''.join(complement[b] for b in reversed(seq))

# Convert to Python lists
seqs = sequences[0].tolist()
labs = labels[0].tolist()

aug_seqs = []
aug_labels = []

for seq, lab in zip(seqs, labs):
    aug_seqs.append(seq)
    aug_labels.append(lab)

    # add reverse complement
    aug_seqs.append(reverse_complement(seq))
    aug_labels.append(lab)

# Convert back to DataFrame (optional)
aug_sequences = pd.DataFrame(aug_seqs, columns=["sequence"])
aug_labels = pd.DataFrame(aug_labels, columns=["label"])

print(aug_sequences.shape)
print(aug_labels.shape)

(572328, 1)
(572328, 1)


In [ ]:
# one-hot encoding
def one_hot_encode(sequence):

    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0 ,0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1]
    }

    vec = np.array([mapping[base] for base in sequence])

    return vec

X_seq = np.array([one_hot_encode(seq) for seq in aug_sequences.iloc[:,0]])
Y_lab = aug_labels.iloc[:,0].values
print(X_seq.shape)
print(Y_lab.shape)

(572328, 200, 4)
(572328,)


In [ ]:
# Splitting our full training dataset into a training and validation set

validation_size = 0.2 #20% used for validation data, 80% used for training data

x_train, x_val, y_train, y_val = train_test_split (
    X_seq, Y_lab, test_size=validation_size, random_state=8, stratify=Y_lab
)

# Need to make sure the labels are 0 - 17 not 1 - 18

y_train = y_train - 1
y_val   = y_val - 1

# One hot encoding my chromatin states


y_train_oh = tf.keras.utils.to_categorical(y_train, 18)
y_val_oh   = tf.keras.utils.to_categorical(y_val, 18)

print(y_train_oh.shape)   # (N, 18)
print(y_train_oh.sum(axis=1)[:5])  # should all be 1.0

(457862, 18)
[1. 1. 1. 1. 1.]


In [ ]:
def attention_pooling_1d(x, name="attn"):
    """
    x: (B, L, C)
    returns: (B, C) attention-weighted pooled features
    """
    # score each position
    scores = tf.keras.layers.Dense(1, name=f"{name}_score")(x)               # (B, L, 1)

    # normalize over positions (axis=1 is the length dimension!)
    weights = tf.keras.layers.Softmax(axis=1, name=f"{name}_softmax")(scores) # (B, L, 1)

    # weight the features
    xw = tf.keras.layers.Multiply(name=f"{name}_mul")([x, weights])          # (B, L, C)

    # sum across positions -> (B, C)
    pooled = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1),
                                    name=f"{name}_sum")(xw)
    return pooled

In [ ]:
# Note to self: 200bp sequences, 4 options per bp, 18 possible chromatin states

# Building the CNN Model:

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(200, 4)),

    # First CNN block (goal: learn simple motifs)
    tf.keras.layers.Conv1D(64, 30, padding="same", activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    # tf.keras.layers.MaxPool1D(2), #Try removing this
    tf.keras.layers.Dropout(0.2),

    # Second CNN block - finds more specific patterns
    tf.keras.layers.Conv1D(128, 15, padding="same", activation="relu", dilation_rate=2, kernel_regularizer=regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.MaxPool1D(2), #Try removing this


    # Third CNN block - finds more specific patterns
    tf.keras.layers.Conv1D(256, 7, padding="same", activation="relu", dilation_rate=4, kernel_regularizer=regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.MaxPool1D(2),


    # Fourth CNN block - to increase the accuracy
    tf.keras.layers.Conv1D(512, 3, padding="same", activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.MaxPool1D(2),


    # Pool the whole thing into one weight matrix
    tf.keras.layers.GlobalAveragePooling1D(),

    # Decision making layer
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    # Now, outputs
    tf.keras.layers.Dense(18, activation="softmax"),

])


model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 200, 64)        │         7,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 200, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 200, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 200, 128)       │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 200, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 200, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_14 (Conv1D)              │ (None, 100, 256)       │       229,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 100, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 100, 256)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_15 (Conv1D)              │ (None, 50, 512)        │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 50, 512)        │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ (None, 50, 512)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 25, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 18)             │         4,626 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 893,906 (3.41 MB)

 Trainable params: 891,986 (3.40 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [ ]:
# Creates the reverse complement of a sequence

def reverse_complement_onehot(x):
    # x: (N, 200, 4) with channels A,C,G,T
    x_rc = x[:, ::-1, :]   # reverse positions
    x_rc = x_rc[:, :, ::-1]  # swap channels A<->T, C<->G
    return x_rc

In [ ]:
# Callbacks makes sure to reset the weights after running the model

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5), # Stop training once you reach high accuracy
    tf.keras.callbacks.ModelCheckpoint("best.keras", save_best_only=True),
]

history = model.fit(
    x_train, y_train_oh,
    validation_data=(x_val, y_val_oh),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
)

val_loss, val_acc = model.evaluate(x_val, y_val_oh)
print("Validation accuracy is " + str(val_acc) + ".")

# Now trying with reverse complement enforcement

x_val_rc = reverse_complement_onehot(x_val)
p_val_fwd = model.predict(x_val, batch_size=256)
p_val_rc  = model.predict(x_val_rc, batch_size=256)
p_val_avg = 0.5 * (p_val_fwd + p_val_rc)
y_true = np.argmax(y_val_oh, axis=1)
y_pred = np.argmax(p_val_avg, axis=1)
val_acc_rc = (y_true == y_pred).mean()
print("RC-averaged validation accuracy:", val_acc_rc)

Epoch 1/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 92s 46ms/step - accuracy: 0.1271 - loss: 2.7714 - val_accuracy: 0.1544 - val_loss: 2.6735 - learning_rate: 5.0000e-04
Epoch 2/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 72s 40ms/step - accuracy: 0.1578 - loss: 2.6589 - val_accuracy: 0.1611 - val_loss: 2.6423 - learning_rate: 5.0000e-04
Epoch 3/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 72s 40ms/step - accuracy: 0.1652 - loss: 2.6301 - val_accuracy: 0.1586 - val_loss: 2.6525 - learning_rate: 5.0000e-04
Epoch 4/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 71s 40ms/step - accuracy: 0.1700 - loss: 2.6198 - val_accuracy: 0.1733 - val_loss: 2.6076 - learning_rate: 5.0000e-04
Epoch 5/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 71s 40ms/step - accuracy: 0.1715 - loss: 2.6124 - val_accuracy: 0.1671 - val_loss: 2.6155 - learning_rate: 5.0000e-04
Epoch 6/50
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 71s 40ms/step - accuracy: 0.1738 - loss: 2.6077 - val_accuracy: 0.1770 - val_loss: 2.6049 - learning_rate: 5.0000e-04
Epoch 7/50
1789/1789 ━━━━━━━━━━━━━━━━━━━

In [ ]:
test_seq =  np.array([one_hot_encode(seq) for seq in test.iloc[:,0]])
test_seq_rc = reverse_complement_onehot(test_seq)

p_fwd = model.predict(test_seq, batch_size=256)
p_rc = model.predict(test_seq_rc, batch_size=256)
pred_probs = 0.5 * (p_fwd + p_rc)

pred_probs_A = np.load("pred_probs_modelA.npy")
print(pred_probs_A.shape)

# pred_probs_comb = (pred_probs + pred_probs_A) / 2

pred_labels = np.argmax(pred_probs, axis=1) + 1

submission = pd.DataFrame(pred_labels)
submission.to_csv("predictions.csv", index=False, header=False)